In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/Sakina152/mm-dg-healthcare.git"
REPO_DIR = "/content/mm-dg-healthcare"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!git log -1 --oneline

Mounted at /content/drive
Working directory: /content/mm-dg-healthcare
08b333f (HEAD -> main, origin/main, origin/HEAD) Add multi-dataset compatibility audit


In [2]:
from pathlib import Path

drive_backup = Path("/content/drive/MyDrive/mm-dg-healthcare-data")
archive = drive_backup / "ptbxl.tar"

print("Drive backup folder exists:", drive_backup.exists())
print("PTB-XL archive exists:", archive.exists())

if archive.exists():
    print(f"Archive size: {archive.stat().st_size / (1024**3):.3f} GB")
    print("Archive path:", archive)
else:
    print("PTB-XL archive was not found at the expected backup path.")

print("\nOther files in the backup folder:")
if drive_backup.exists():
    for p in sorted(drive_backup.iterdir()):
        print(p.name, f"({p.stat().st_size / (1024**2):.1f} MB)" if p.is_file() else "(folder)")

Drive backup folder exists: True
PTB-XL archive exists: True
Archive size: 0.541 GB
Archive path: /content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar

Other files in the backup folder:
configs (folder)
ptbxl (folder)
ptbxl.tar (554.1 MB)
results_backup (folder)


In [3]:
import tarfile
from pathlib import Path

archive = Path("/content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar")

with tarfile.open(archive, "r:*") as tar:
    members = tar.getnames()

print("Total archive entries:", len(members))
print("\nFirst 20 entries:")
for name in members[:20]:
    print(name)

print("\nMetadata files in archive:")
for name in members:
    if name.endswith(("ptbxl_database.csv", "scp_statements.csv")):
        print(name)

Total archive entries: 43648

First 20 entries:
ptbxl
ptbxl/scp_statements.csv
ptbxl/records100
ptbxl/records100/13000
ptbxl/records100/13000/13674_lr.dat
ptbxl/records100/13000/13541_lr.hea
ptbxl/records100/13000/13950_lr.dat
ptbxl/records100/13000/13065_lr.dat
ptbxl/records100/13000/13390_lr.hea
ptbxl/records100/13000/13239_lr.hea
ptbxl/records100/13000/13554_lr.hea
ptbxl/records100/13000/13274_lr.dat
ptbxl/records100/13000/13260_lr.hea
ptbxl/records100/13000/13083_lr.hea
ptbxl/records100/13000/13324_lr.hea
ptbxl/records100/13000/13790_lr.hea
ptbxl/records100/13000/13861_lr.hea
ptbxl/records100/13000/13216_lr.dat
ptbxl/records100/13000/13629_lr.hea
ptbxl/records100/13000/13581_lr.hea

Metadata files in archive:
ptbxl/scp_statements.csv
ptbxl/ptbxl_database.csv


In [4]:
from pathlib import Path
import tarfile
import pandas as pd

REPO = Path("/content/mm-dg-healthcare")
ARCHIVE = Path("/content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar")
RAW = REPO / "data" / "raw"
PTB_DIR = RAW / "ptbxl"

assert REPO.exists(), f"Repository not found: {REPO}"
assert ARCHIVE.exists(), f"Backup archive not found: {ARCHIVE}"

db_path = PTB_DIR / "ptbxl_database.csv"
scp_path = PTB_DIR / "scp_statements.csv"

if db_path.exists() and scp_path.exists():
    print("Both metadata files already exist. Skipping extraction.")
else:
    RAW.mkdir(parents=True, exist_ok=True)

    with tarfile.open(ARCHIVE, "r:*") as tar:
        # Validate paths before extraction.
        for member in tar.getmembers():
            target = (RAW / member.name).resolve()
            if not target.is_relative_to(RAW.resolve()):
                raise ValueError(f"Unsafe archive path: {member.name}")

        tar.extractall(RAW)

    print("Archive extracted.")

assert db_path.exists(), f"Missing metadata: {db_path}"
assert scp_path.exists(), f"Missing metadata: {scp_path}"

ptb = pd.read_csv(db_path)
scp = pd.read_csv(scp_path, index_col=0)

print("\nPTB-XL restored successfully!")
print("Dataset directory:", PTB_DIR)
print("Metadata shape:", ptb.shape)
print("SCP statements shape:", scp.shape)
print("ECG records in metadata:", len(ptb))
print("Required metadata files verified.")

/tmp/ipykernel_1092/2075355843.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(RAW)


Archive extracted.

PTB-XL restored successfully!
Dataset directory: /content/mm-dg-healthcare/data/raw/ptbxl
Metadata shape: (21799, 28)
SCP statements shape: (71, 12)
ECG records in metadata: 21799
Required metadata files verified.


Set up paths and locate PTB-XL

In [5]:

from pathlib import Path
import ast
import pandas as pd
import json
import os

REPO = Path("/content/mm-dg-healthcare")
RAW = REPO / "data" / "raw"

assert REPO.exists(), (
    "Project repository not found. "
    "Run your existing Colab setup notebook first."
)

matches = list(RAW.rglob("ptbxl_database.csv"))
assert matches, (
    "PTB-XL metadata not found. Check extraction or restore "
    "the dataset from your Google Drive backup."
)

PTB_DIR = matches[0].parent
db_path = PTB_DIR / "ptbxl_database.csv"
scp_path = PTB_DIR / "scp_statements.csv"

ptb = pd.read_csv(db_path)
scp = pd.read_csv(scp_path, index_col=0)

print("Repository:", REPO)
print("PTB-XL directory:", PTB_DIR)
print("Metadata shape:", ptb.shape)
print("Unique patients:", ptb["patient_id"].nunique())
print("Columns:", ptb.columns.tolist())

display(ptb.head())
display(scp.head())


Repository: /content/mm-dg-healthcare
PTB-XL directory: /content/mm-dg-healthcare/data/raw/ptbxl
Metadata shape: (21799, 28)
Unique patients: 18869
Columns: ['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


Audit signal files and demographic metadata

In [6]:

print("Age missing (%):", round(ptb["age"].isna().mean() * 100, 2))
print("Sex missing (%):", round(ptb["sex"].isna().mean() * 100, 2))

print("\nAge statistics:")
display(ptb["age"].describe())

print("\nSex distribution:")
display(ptb["sex"].value_counts(dropna=False))

signal_col = "filename_lr"

if signal_col in ptb.columns:
    signal_paths = ptb[signal_col].dropna().astype(str)

    data_found = sum(
        (PTB_DIR / p).with_suffix(".dat").exists()
        for p in signal_paths
    )
    headers_found = sum(
        (PTB_DIR / p).with_suffix(".hea").exists()
        for p in signal_paths
    )

    print("\nExpected signal files:", len(signal_paths))
    print("Signal files found:", data_found)
    print("Header files found:", headers_found)

if "site" in ptb.columns:
    print("\nRecords per site:")
    display(ptb["site"].value_counts(dropna=False))


Age missing (%): 0.0
Sex missing (%): 0.0

Age statistics:


,age
count,21799.000000
mean,62.769301
std,32.308813
min,2.000000
25%,50.000000
50%,62.000000
75%,72.000000
max,300.000000



Sex distribution:


,count
sex,
0,11354
1,10445



Expected signal files: 21799
Signal files found: 21799
Header files found: 21799

Records per site:


,count
site,
0.0,8940
1.0,6294
2.0,5075
3.0,576
4.0,100
5.0,58
6.0,46
8.0,43
7.0,43


Audit PTB-XL diagnostic labels

In [7]:

assert "scp_codes" in ptb.columns
assert "diagnostic" in scp.columns
assert "diagnostic_class" in scp.columns

diagnostic_map = scp.loc[
    scp["diagnostic"].fillna(0).astype(int) == 1,
    "diagnostic_class"
].dropna().to_dict()

def parse_scp_codes(value):
    if pd.isna(value):
        return {}
    try:
        return ast.literal_eval(value) if isinstance(value, str) else value
    except (ValueError, SyntaxError):
        return {}

counts = {label: 0 for label in ["NORM", "MI", "STTC", "CD", "HYP"]}

for value in ptb["scp_codes"]:
    codes = parse_scp_codes(value)
    classes = {
        diagnostic_map[code]
        for code in codes
        if code in diagnostic_map
    }

    for label in counts:
        if label in classes:
            counts[label] += 1

label_audit = pd.DataFrame({
    "diagnostic_superclass": list(counts.keys()),
    "record_count": list(counts.values())
})

display(label_audit)


,diagnostic_superclass,record_count
0,NORM,9514
1,MI,5469
2,STTC,5235
3,CD,4898
4,HYP,2649


Inventory the candidate datasets

In [8]:

candidates = {
    "PTB-XL": ["ptbxl_database.csv"],
    "Chapman-Shaoxing-Ningbo": [
        "ConditionNames_SNOMED-CT.csv",
        "Diagnostics.xlsx",
        "RECORDS"
    ],
    "CPSC 2018": ["REFERENCE.csv", "RECORDS"],
    "Georgia 12-lead ECG": ["RECORDS", "REFERENCE.csv"],
    "MIMIC-IV-ECG": [
        "record-list.csv",
        "machine_measurements.csv"
    ]
}

inventory = []

for name, markers in candidates.items():
    found = []

    for marker in markers:
        for path in RAW.rglob(marker):
            if name == "PTB-XL" and path.name != "ptbxl_database.csv":
                continue
            found.append(str(path))

    inventory.append({
        "dataset": name,
        "found_locally": bool(found),
        "example_paths": "; ".join(sorted(set(found))[:5])
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)


,dataset,found_locally,example_paths
0,PTB-XL,True,/content/mm-dg-healthcare/data/raw/ptbxl/ptbxl...
1,Chapman-Shaoxing-Ningbo,False,
2,CPSC 2018,False,
3,Georgia 12-lead ECG,False,
4,MIMIC-IV-ECG,False,


Save audit results

In [9]:

OUT = REPO / "experiments" / "results" / "dataset_compatibility_audit"
OUT.mkdir(parents=True, exist_ok=True)

label_audit.to_csv(
    OUT / "ptbxl_label_counts.csv", index=False
)
inventory_df.to_csv(
    OUT / "candidate_dataset_inventory.csv", index=False
)

ptb_summary = pd.DataFrame([{
    "dataset": "PTB-XL",
    "records": len(ptb),
    "unique_patients": ptb["patient_id"].nunique(),
    "age_missing_pct": ptb["age"].isna().mean() * 100,
    "sex_missing_pct": ptb["sex"].isna().mean() * 100,
}])

ptb_summary.to_csv(
    OUT / "ptbxl_metadata_audit.csv", index=False
)

print("Audit reports saved in:", OUT)
print(*[p.name for p in OUT.glob("*.csv")], sep="\n")


Audit reports saved in: /content/mm-dg-healthcare/experiments/results/dataset_compatibility_audit
ptbxl_label_counts.csv
candidate_dataset_inventory.csv
ptbxl_metadata_audit.csv


Back up audit CSVs to Google Drive

In [10]:

from pathlib import Path
import shutil

# Mount Google Drive if it is not already mounted
from google.colab import drive
drive.mount("/content/drive")

# Source and backup directories
SOURCE = Path(
    "/content/mm-dg-healthcare/experiments/results/"
    "dataset_compatibility_audit"
)

BACKUP = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/"
    "results_backup/dataset_compatibility_audit"
)

assert SOURCE.exists(), f"Audit output folder not found: {SOURCE}"

# Copy all CSV reports to Drive
BACKUP.mkdir(parents=True, exist_ok=True)
csv_files = list(SOURCE.glob("*.csv"))

assert csv_files, f"No CSV files found in {SOURCE}"

for src in csv_files:
    dest = BACKUP / src.name
    shutil.copy2(src, dest)
    print(f"Backed up: {src.name} -> {dest}")

# Verify the backups
print("\nVerification:")
for src in csv_files:
    dest = BACKUP / src.name
    assert dest.exists(), f"Backup missing: {dest}"
    assert dest.stat().st_size == src.stat().st_size, (
        f"File size mismatch: {src.name}"
    )
    print(f"OK: {dest.name} ({dest.stat().st_size:,} bytes)")

print(f"\nAll {len(csv_files)} CSV reports backed up to:\n{BACKUP}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backed up: ptbxl_label_counts.csv -> /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/dataset_compatibility_audit/ptbxl_label_counts.csv
Backed up: candidate_dataset_inventory.csv -> /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/dataset_compatibility_audit/candidate_dataset_inventory.csv
Backed up: ptbxl_metadata_audit.csv -> /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/dataset_compatibility_audit/ptbxl_metadata_audit.csv

Verification:
OK: ptbxl_label_counts.csv (80 bytes)
OK: candidate_dataset_inventory.csv (203 bytes)
OK: ptbxl_metadata_audit.csv (91 bytes)

All 3 CSV reports backed up to:
/content/drive/MyDrive/mm-dg-healthcare-data/results_backup/dataset_compatibility_audit


Load PTB-XL metadata and inspect diagnostic labels

In [11]:

from pathlib import Path
import ast
import pandas as pd

REPO = Path("/content/mm-dg-healthcare")
PTB_DIR = REPO / "data" / "raw" / "ptbxl"
OUT = REPO / "experiments" / "results" / "dataset_compatibility_audit"
OUT.mkdir(parents=True, exist_ok=True)

db_path = PTB_DIR / "ptbxl_database.csv"
scp_path = PTB_DIR / "scp_statements.csv"

assert db_path.exists(), f"Missing: {db_path}"
assert scp_path.exists(), f"Missing: {scp_path}"

df = pd.read_csv(db_path)
scp = pd.read_csv(scp_path, index_col=0)

print("PTB-XL records:", len(df))
print("SCP statement rows:", len(scp))
print("Metadata columns:", df.columns.tolist())
print("SCP mapping columns:", scp.columns.tolist())

assert "scp_codes" in df.columns
assert "diagnostic" in scp.columns
assert "diagnostic_class" in scp.columns


PTB-XL records: 21799
SCP statement rows: 71
Metadata columns: ['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']
SCP mapping columns: ['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']


Count official diagnostic superclasses

In [12]:

# Keep statements marked as diagnostic in the official SCP mapping.
diagnostic = scp[scp["diagnostic"] == 1].copy()

print("Diagnostic SCP statements:", len(diagnostic))
print("\nSuperclass counts among diagnostic SCP statements:")
display(
    diagnostic["diagnostic_class"]
    .value_counts(dropna=False)
    .rename_axis("diagnostic_class")
    .reset_index(name="statement_count")
)

def get_diagnostic_classes(value):
    codes = ast.literal_eval(value) if isinstance(value, str) else {}
    classes = set()

    for code, likelihood in codes.items():
        if likelihood <= 0 or code not in diagnostic.index:
            continue

        superclass = diagnostic.loc[code, "diagnostic_class"]

        if pd.notna(superclass) and str(superclass).strip():
            classes.add(str(superclass).strip())

    return sorted(classes)

df["diagnostic_classes"] = df["scp_codes"].apply(get_diagnostic_classes)

class_counts = (
    df["diagnostic_classes"]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("diagnostic_class")
    .reset_index(name="record_count")
)

print("\nRecord counts by diagnostic superclass:")
display(class_counts)

class_counts.to_csv(
    OUT / "ptbxl_diagnostic_superclass_counts.csv",
    index=False
)

print("Saved diagnostic superclass counts.")


Diagnostic SCP statements: 44

Superclass counts among diagnostic SCP statements:


,diagnostic_class,statement_count
0,MI,14
1,STTC,13
2,CD,11
3,HYP,5
4,NORM,1



Record counts by diagnostic superclass:


,diagnostic_class,record_count
0,NORM,9514
1,MI,5469
2,STTC,5108
3,CD,4898
4,HYP,2649


Saved diagnostic superclass counts.


Save a complete PTB-XL diagnostic audit

In [13]:

# Summarize the diagnostic classes assigned to each record.
df["num_diagnostic_classes"] = df["diagnostic_classes"].apply(len)

record_summary = pd.DataFrame({
    "record_id": df["ecg_id"],
    "num_diagnostic_classes": df["num_diagnostic_classes"],
    "diagnostic_classes": df["diagnostic_classes"].apply(
        lambda classes: ";".join(classes)
    ),
})

record_summary.to_csv(
    OUT / "ptbxl_record_diagnostic_classes.csv",
    index=False
)

summary = pd.DataFrame({
    "metric": [
        "total_records",
        "records_with_diagnostic_class",
        "records_without_diagnostic_class",
        "records_with_multiple_diagnostic_classes",
    ],
    "value": [
        len(df),
        int((df["num_diagnostic_classes"] > 0).sum()),
        int((df["num_diagnostic_classes"] == 0).sum()),
        int((df["num_diagnostic_classes"] > 1).sum()),
    ],
})

display(summary)

summary.to_csv(
    OUT / "ptbxl_diagnostic_audit_summary.csv",
    index=False
)

print("Saved record-level diagnostic mapping and audit summary.")
print("Output folder:", OUT)


,metric,value
0,total_records,21799
1,records_with_diagnostic_class,21375
2,records_without_diagnostic_class,424
3,records_with_multiple_diagnostic_classes,5068


Saved record-level diagnostic mapping and audit summary.
Output folder: /content/mm-dg-healthcare/experiments/results/dataset_compatibility_audit


Check all audit CSV reports

In [14]:

from pathlib import Path

OUT = Path(
    "/content/mm-dg-healthcare/experiments/results/"
    "dataset_compatibility_audit"
)

csv_files = sorted(OUT.glob("*.csv"))

print("Audit CSV reports:", len(csv_files))
for path in csv_files:
    print(f"{path.name}: {path.stat().st_size:,} bytes")

assert csv_files, "No audit CSV files found."


Audit CSV reports: 6
candidate_dataset_inventory.csv: 203 bytes
ptbxl_diagnostic_audit_summary.csv: 152 bytes
ptbxl_diagnostic_superclass_counts.csv: 75 bytes
ptbxl_label_counts.csv: 80 bytes
ptbxl_metadata_audit.csv: 91 bytes
ptbxl_record_diagnostic_classes.csv: 278,607 bytes


Back up all audit CSVs to Google Drive

In [15]:

from pathlib import Path
import shutil
from google.colab import drive

# Mount Google Drive. If already mounted, Colab may say it is mounted.
drive.mount("/content/drive")

SOURCE = Path(
    "/content/mm-dg-healthcare/experiments/results/"
    "dataset_compatibility_audit"
)

BACKUP = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/"
    "results_backup/dataset_compatibility_audit"
)

assert SOURCE.exists(), f"Audit output folder not found: {SOURCE}"

BACKUP.mkdir(parents=True, exist_ok=True)

csv_files = sorted(SOURCE.glob("*.csv"))
assert csv_files, f"No CSV files found in {SOURCE}"

for src in csv_files:
    dest = BACKUP / src.name
    shutil.copy2(src, dest)

    assert dest.exists(), f"Backup missing: {dest}"
    assert dest.stat().st_size == src.stat().st_size, (
        f"Backup size mismatch: {src.name}"
    )

    print(f"Verified backup: {dest.name}")

print(f"\nSuccessfully verified {len(csv_files)} CSV backups.")
print("Google Drive destination:", BACKUP)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verified backup: candidate_dataset_inventory.csv
Verified backup: ptbxl_diagnostic_audit_summary.csv
Verified backup: ptbxl_diagnostic_superclass_counts.csv
Verified backup: ptbxl_label_counts.csv
Verified backup: ptbxl_metadata_audit.csv
Verified backup: ptbxl_record_diagnostic_classes.csv

Successfully verified 6 CSV backups.
Google Drive destination: /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/dataset_compatibility_audit


In [16]:

from pathlib import Path

BACKUP = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/"
    "results_backup/dataset_compatibility_audit"
)

for path in sorted(BACKUP.glob("*.csv")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

print("\nDrive audit backup verification complete.")


candidate_dataset_inventory.csv: 203 bytes
ptbxl_diagnostic_audit_summary.csv: 152 bytes
ptbxl_diagnostic_superclass_counts.csv: 75 bytes
ptbxl_label_counts.csv: 80 bytes
ptbxl_metadata_audit.csv: 91 bytes
ptbxl_record_diagnostic_classes.csv: 278,607 bytes

Drive audit backup verification complete.
